# M2 — Adaptive Question Generator (FLAN-T5-Large + LoRA)
**Do this last** — heaviest model, ~770M params. LoRA freezes the base and
trains small adapter matrices (<1% of params) so it fits a free T4's 16GB VRAM.
Target: **BLEU-4 >= 0.35, ROUGE-L >= 0.50**. See `docs/phases/MENTORA_Phase4_Model_Training.md` §6.

Schema (`input_text`, `target_text` — target is a JSON string) matches the
plan doc exactly, no adaptation needed. One real caveat carried over from
Phase 3: `target_text`'s `correct_answer` field is `null` for all 1777
MDCAT-derived rows (see `mentora_data/SOURCES.md` — the answer-key OCR never
worked out) and real for the 378 Math rows. The model will still learn to
generate well-formed questions+options either way, but won't have seen a
correct answer to imitate for most of its training data.

## Setup — mount Drive, confirm GPU, install deps
Same first 3 cells in every Mentora training notebook. See `docs/phases/MENTORA_Phase4_Model_Training.md` §0.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Nov  4 06:25:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [3]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

In [4]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121 --break-system-packages
!pip install -q transformers peft accelerate datasets evaluate sacrebleu rouge_score --break-system-packages

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Load base model + apply LoRA

In [5]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
!pip install -q -U torchao

In [7]:
import transformers, peft
print(transformers.__version__, peft.__version__)

5.22.0 0.21.2


In [14]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset

tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-large')
import torch

base_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large', torch_dtype=torch.float32)

lora_config = LoraConfig(
    # task_type=TaskType.SEQ_2_SEQ_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q", "v"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # expect well under 1% trainable

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 2,359,296 || all params: 785,509,376 || trainable%: 0.3004


## Load and tokenize the Phase 3 data

In [9]:
df = pd.read_csv(f'{DATA}/processed/m2/flan_t5_training_data.csv')
print(df.shape)
ds = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=96)
    targets = tokenizer(batch['target_text'], truncation=True, padding='max_length', max_length=256)
    # Mask padding positions with -100 so CrossEntropyLoss ignores them;
    # without this the model trivially learns to predict PAD for most of
    # each 256-token target, collapsing training_loss to ~0 while never
    # learning actual question content (root cause of the first run's
    # training_loss=0.0 / eval_loss=nan / near-zero BLEU-4/ROUGE-L).
    labels = targets['input_ids']
    labels = [[(tok if tok != tokenizer.pad_token_id else -100) for tok in seq] for seq in labels]
    inputs['labels'] = labels
    return inputs

ds = ds.map(tokenize, batched=True)

(2155, 3)


Map:   0%|          | 0/1939 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

## Metrics (BLEU-4 + ROUGE-L)

In [10]:
import evaluate
import numpy as np
sacrebleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rougeL = rouge.compute(predictions=decoded_preds, references=decoded_labels)["rougeL"]
    return {"bleu4": bleu["score"], "rougeL": rougeL}

## Train — checkpoint by STEPS not epochs (a 3-epoch run on a free T4 can span multiple sessions)

> **Before re-running from here:** three issues found across attempts 1-2. (1) `tokenize()` two cells up now masks padding tokens to `-100` (was missing). (2) T5/FLAN-T5 is numerically unstable under fp16 (training_loss stuck at 0.0, eval_loss nan) -- `fp16=False` now, see notebooks/README.md for sources. (1)+(2) together fixed the 0.0/nan symptom in attempt 2 (training_loss=8.53, eval_loss=1.76, both real numbers) but BLEU-4/ROUGE-L barely moved (0.085/0.157). (3) the eval step's `.generate()` call was defaulting to `max_length=21` -- truncating every prediction to 21 tokens against a target tokenized up to 256, which alone would wreck BLEU/ROUGE regardless of model quality. Now set via `generation_max_length=256` above. The cell below also auto-resumes from any existing checkpoint under `mentora_models/model2_question_generator/checkpoint-*` on Drive -- those checkpoints are from the buggy runs. **Delete that folder on Drive before re-running this notebook**, so training starts fresh.

In [13]:
import shutil, glob
old_ckpts = glob.glob(f'{OUT_DIR}/checkpoint-*')
for c in old_ckpts: shutil.rmtree(c)
print(f'deleted {len(old_ckpts)} stale checkpoint(s)' if old_ckpts else 'no checkpoints to delete')

deleted 2 stale checkpoint(s)


In [15]:
OUT_DIR = f'{MODELS}/model2_question_generator'
args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch size 16
    num_train_epochs=3,
    learning_rate=1e-4,
    predict_with_generate=True,
    generation_max_length=256,  # target_text is tokenized up to 256 tokens
                                # (see tokenize() above) -- without this,
                                # eval generate() defaults to ~20 tokens,
                                # truncating every prediction and wrecking
                                # BLEU-4/ROUGE-L regardless of model quality.
    fp16=False,  # T5/FLAN-T5 is well-documented as numerically unstable
                 # under fp16 (training_loss stuck at 0.0, eval_loss NaN,
                 # independent of the label-masking bug fixed above) --
                 # see notebooks/README.md. T4 also lacks efficient native
                 # bf16, so plain fp32 is the safe choice here.
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

import glob
existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Bleu4,Rougel
200,No log,1.756476,11.804570,0.149016
366,No log,1.719468,12.127204,0.172256


TrainOutput(global_step=366, training_loss=8.520651228440915, metrics={'train_runtime': 2255.1478, 'train_samples_per_second': 2.579, 'train_steps_per_second': 0.162, 'total_flos': 2521686632693760.0, 'train_loss': 8.520651228440915, 'epoch': 3.0})

## Save — LoRA adapter only (small, tens of MB), NOT merged into the base yet
Merging happens once, in Phase 5, right before ONNX export — keeping them separate here means you can keep iterating without re-touching the much larger base model.

In [16]:
model.save_pretrained(f'{OUT_DIR}/lora_adapter')
tokenizer.save_pretrained(f'{OUT_DIR}/lora_adapter')
print(trainer.evaluate())

Training Loss,Validation Loss,Step,Bleu4,Rougel
No log,1.719468,366,12.127204,0.172256


{'eval_loss': 1.7194679975509644, 'eval_bleu4': 12.127204133669785, 'eval_rougeL': 0.1722555533274824}


## If you miss the target (BLEU-4 < 0.35 or ROUGE-L < 0.50)
- Check `target_text` is consistently formatted JSON for every row — already
  validated once in Phase 3 (`verify_phase3.py`), but worth re-checking if
  you've touched the data since
- Try `r=16` in the LoRA config for more adapter capacity
- The dataset (2155 examples) clears the doc's 500-minimum comfortably;
  growing the M1 question bank further (which M2 reuses) is still the
  highest-leverage fix if quality is the issue, not quantity